In [0]:
display(
    dbutils.fs.ls(
        "abfss://rawdata@manuelstorage01.dfs.core.windows.net/"
    )
)

In [0]:
import time
import requests
from pyspark.sql import SparkSession

# 1. Configuration & API Setup
BASE_URL = "https://api.openbrewerydb.org/v1/breweries"
PER_PAGE = 200  # Max records allowed per page by OpenBreweryDB
page = 1
all_breweries = []

print("Starting OpenBreweryDB ingestion...")

# 2. Paginate through all records
while True:
    response = requests.get(
        BASE_URL, params={"page": page, "per_page": PER_PAGE}
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Failed to fetch page {page}. Status code: {response.status_code}"
        )

    data = response.json()

    # API returns empty list [] when past the last page
    if not data:
        break

    all_breweries.extend(data)
    print(f"Fetched page {page} ({len(data)} records)")
    page += 1
    time.sleep(0.1)  # API polite delay

print(f"Total breweries ingested: {len(all_breweries)}")

# 3. Convert to PySpark DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Build schema from response documentation.
# Prevents LongType and DoubleType collision on lattitude and longitude
schema = StructType([
    StructField('id', StringType(), True),
    StructField('name', StringType(), True),
    StructField('brewery_type', StringType(), True),
    StructField('street', StringType(), True),
    StructField('address_2', StringType(), True),
    StructField('address_3', StringType(), True),
    StructField('city', StringType(), True),
    StructField('state', StringType(), True),
    StructField('county_province', StringType(), True),
    StructField('postal_code', StringType(), True),
    StructField('country', StringType(), True),
    StructField('longitude', DoubleType(), True),
    StructField('latitude', DoubleType(), True),
    StructField('phone', StringType(), True),
    StructField('website_url', StringType(), True),
    StructField('updated_at', StringType(), True),
    StructField('created_at', StringType(), True)
])
df = spark.createDataFrame(all_breweries, schema=schema)


# 4. Peek first 5 rows of the DataFrame
display(df.limit(5))

# 5. Save to Azure Blob Storage under Bronze/Breweries
abfss_path = "abfss://rawdata@manuelstorage01.dfs.core.windows.net/Bronze/Breweries"

(
    df.coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(abfss_path)
)

print(f"Successfully saved CSV with headers to: {abfss_path}")